# FitNova v5 — Phase 4: SSL Pretrain (Masked-Joint Reconstruction)

Pretrains the ST-GCN encoder on the full Fit3D MediaPipe corpus using
masked-joint autoencoding. The encoder weights are then loaded into the
supervised model in Phase 5.

**Cost:** ~25-45 min on L4 / ~15-25 min on A100. ~0.7 Colab compute units.

**Inputs (must be uploaded to MyDrive/fitnova_v5/):**
- `fitnova_v5_src.zip` — backend source code
- `v5_dataset.zip` — pre-built v5 dataset

**Outputs (saved to MyDrive/fitnova_v5_results/):**
- `ssl_encoder.weights.h5` — pretrained ST-GCN encoder weights
- `ssl_history.json` — per-epoch loss history
- `ssl_checkpoints/epoch_NN.weights.h5` — per-epoch checkpoints (crash-resume)

**Crash-resume:** If Colab kills the runtime mid-training, just rerun every
cell from the top — the training cell auto-detects the latest checkpoint and
resumes from the next epoch.


In [ ]:
# ── GPU + Drive mount ────────────────────────────────────────────────────────
import os, sys, json, time, shutil
import tensorflow as tf
print("TF:    ", tf.__version__)
print("Keras: ", tf.keras.__version__)
print("GPUs:  ", tf.config.list_physical_devices("GPU"))

from google.colab import drive
drive.mount("/content/drive")

DRIVE_BASE  = "/content/drive/MyDrive/fitnova_v5"
RESULTS_DIR = "/content/drive/MyDrive/fitnova_v5_results"
os.makedirs(DRIVE_BASE,  exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print()
print("DRIVE_BASE :", DRIVE_BASE)
print("RESULTS_DIR:", RESULTS_DIR)


In [ ]:
# ── Extract backend source from Drive ────────────────────────────────────────
SRC_ZIP  = f"{DRIVE_BASE}/fitnova_v5_src.zip"
WORK_DIR = "/content/fitnova"

assert os.path.isfile(SRC_ZIP), (
    f"Source zip not found at {SRC_ZIP}.\n"
    f"Build it locally with: python _build_v5_colab_bundle.py\n"
    f"Then upload fitnova_v5_src.zip to MyDrive/fitnova_v5/"
)

if not os.path.isdir(f"{WORK_DIR}/backend"):
    os.makedirs(WORK_DIR, exist_ok=True)
    !unzip -q "$SRC_ZIP" -d "$WORK_DIR"

sys.path.insert(0, WORK_DIR)
print("Source ready at", WORK_DIR)
print("Has backend:    ", os.path.exists(f"{WORK_DIR}/backend"))
print("Has st_gcn:     ", os.path.exists(f"{WORK_DIR}/backend/training/models/st_gcn.py"))


In [ ]:
# ── Extract pre-built v5 dataset from Drive ─────────────────────────────────
DATA_ZIP = f"{DRIVE_BASE}/v5_dataset.zip"
DATA_DIR = "/content/v5_dataset"

assert os.path.isfile(DATA_ZIP), (
    f"Dataset zip not found at {DATA_ZIP}.\n"
    f"Build it locally with: python _build_v5_colab_bundle.py\n"
    f"Then upload v5_dataset.zip to MyDrive/fitnova_v5/"
)

if not os.path.isdir(DATA_DIR) or not os.path.isfile(f"{DATA_DIR}/train.npz"):
    os.makedirs(DATA_DIR, exist_ok=True)
    !unzip -q "$DATA_ZIP" -d "$DATA_DIR"

print("Dataset files:", sorted(os.listdir(DATA_DIR)))

with open(f"{DATA_DIR}/dataset_info.json") as f:
    DATASET_INFO = json.load(f)
for k in ("n_exercises", "n_joint_groups", "target_frames", "n_canonical_joints", "n_angular"):
    print(f"  {k:<22s}: {DATASET_INFO[k]}")


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
from tensorflow import keras
from backend.training.models.st_gcn import build_v5_ssl_model

SSL_DIR     = f"{RESULTS_DIR}/ssl_checkpoints"
SSL_HISTORY = f"{RESULTS_DIR}/ssl_history.json"
SSL_FINAL   = f"{RESULTS_DIR}/ssl_encoder.weights.h5"
os.makedirs(SSL_DIR, exist_ok=True)


In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
EPOCHS       = 20
BATCH_SIZE   = 32
MASK_RATIO   = 0.30
LR           = 1e-3
SEED         = 42

T_FRAMES = DATASET_INFO["target_frames"]
J        = DATASET_INFO["n_canonical_joints"]


In [ ]:
# ── Load training pose data ──────────────────────────────────────────────────
train_npz = np.load(f"{DATA_DIR}/train.npz")
val_npz   = np.load(f"{DATA_DIR}/val.npz")

train_pose = train_npz["pose"].astype(np.float32)   # (N, 64, 15, 4) MediaPipe canonical + visibility
val_pose   = val_npz["pose"].astype(np.float32)
print("train_pose:", train_pose.shape)
print("val_pose:  ", val_pose.shape)
print("finite:    ", np.isfinite(train_pose).all(), np.isfinite(val_pose).all())


In [ ]:
# ── Build masked-joint dataset (random per-joint per-frame mask) ─────────────
def make_ssl_dataset(pose, mask_ratio, batch_size, shuffle, seed):
    """
    Yields (inputs, target_pose_xyz) tuples.

    For each batch we draw a fresh random mask. Masked positions in
    pose_masked have x,y,z zeroed; visibility (channel 3) is left intact so
    the model can still see "this joint was visible before being masked".
    """
    pose = pose.astype(np.float32)
    pose_xyz = pose[..., :3]
    n = len(pose)

    def gen():
        rng = np.random.default_rng(seed)
        order = np.arange(n)
        while True:
            if shuffle:
                rng.shuffle(order)
            for i in range(0, n, batch_size):
                idx  = order[i:i+batch_size]
                p    = pose[idx].copy()
                mask = (rng.random((*p.shape[:3], 1)) < mask_ratio).astype(np.float32)
                p_masked = p.copy()
                p_masked[..., :3] *= (1.0 - mask)
                yield (
                    {"pose_masked": p_masked, "mask": mask},
                    pose_xyz[idx],
                )

    sig = (
        {
            "pose_masked": tf.TensorSpec(shape=(None, T_FRAMES, J, 4), dtype=tf.float32),
            "mask":        tf.TensorSpec(shape=(None, T_FRAMES, J, 1), dtype=tf.float32),
        },
        tf.TensorSpec(shape=(None, T_FRAMES, J, 3), dtype=tf.float32),
    )
    ds = tf.data.Dataset.from_generator(gen, output_signature=sig)
    return ds.prefetch(2)

steps_per_epoch  = max(1, len(train_pose) // BATCH_SIZE)
val_steps        = max(1, len(val_pose)   // BATCH_SIZE)
print(f"steps/epoch: {steps_per_epoch}   val steps: {val_steps}")

train_ds = make_ssl_dataset(train_pose, MASK_RATIO, BATCH_SIZE, shuffle=True,  seed=SEED)
val_ds   = make_ssl_dataset(val_pose,   MASK_RATIO, BATCH_SIZE, shuffle=False, seed=SEED + 1)


In [ ]:
# ── Build model ──────────────────────────────────────────────────────────────
def masked_recon_loss(y_true, y_pred):
    """MSE over masked positions only (skip visible joints).

    NOTE: We don't have access to the mask tensor inside the loss, so we
    just compute mean-MSE over all positions. The masked locations are
    where the model was forced to reconstruct without input, so they
    dominate the gradient anyway.
    """
    return tf.reduce_mean(tf.square(y_true - y_pred))

ssl_model = build_v5_ssl_model(
    target_frames=T_FRAMES,
    n_joints=J,
    n_pose_channels=4,
)
ssl_model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=LR, weight_decay=1e-4, clipnorm=1.0),
    loss={"pose_recon": masked_recon_loss},
)
ssl_model.summary(line_length=120)


In [ ]:
# ── Crash-resume: detect latest checkpoint ───────────────────────────────────
def find_latest_epoch(ckpt_dir):
    if not os.path.isdir(ckpt_dir):
        return 0
    files = [f for f in os.listdir(ckpt_dir) if f.startswith("epoch_") and f.endswith(".weights.h5")]
    if not files:
        return 0
    epochs = [int(f.split("_")[1].split(".")[0]) for f in files]
    return max(epochs)

initial_epoch = find_latest_epoch(SSL_DIR)
if initial_epoch > 0:
    last_ckpt = f"{SSL_DIR}/epoch_{initial_epoch:02d}.weights.h5"
    print(f"Resuming from epoch {initial_epoch} ({last_ckpt})")
    ssl_model.load_weights(last_ckpt)
else:
    print("Fresh start (no prior SSL checkpoints found)")

# History resume
history_acc = {"loss": [], "val_loss": []}
if os.path.isfile(SSL_HISTORY):
    with open(SSL_HISTORY) as f:
        history_acc = json.load(f)
    history_acc.setdefault("loss", [])
    history_acc.setdefault("val_loss", [])
    print(f"Loaded prior history: {len(history_acc['loss'])} epochs")


In [ ]:
# ── Per-epoch checkpoint callback (saves to Drive every epoch) ──────────────
class DriveCheckpoint(keras.callbacks.Callback):
    def __init__(self, ckpt_dir, history_path, history_acc, total_epochs):
        super().__init__()
        self.ckpt_dir = ckpt_dir
        self.history_path = history_path
        self.history_acc  = history_acc
        self.total_epochs = total_epochs

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        # Epoch indexing: Keras `epoch` is 0-based; we want 1-based filenames
        ep1 = epoch + 1
        ckpt = f"{self.ckpt_dir}/epoch_{ep1:02d}.weights.h5"
        self.model.save_weights(ckpt)
        # Accumulate history
        self.history_acc["loss"].append(float(logs.get("loss", 0.0)))
        self.history_acc["val_loss"].append(float(logs.get("val_loss", 0.0)))
        with open(self.history_path, "w") as f:
            json.dump(self.history_acc, f, indent=2)
        # Keep only the most recent 3 checkpoints to save Drive space
        files = sorted(
            f for f in os.listdir(self.ckpt_dir)
            if f.startswith("epoch_") and f.endswith(".weights.h5")
        )
        for f in files[:-3]:
            try:
                os.remove(os.path.join(self.ckpt_dir, f))
            except OSError:
                pass
        print(f"  [saved {os.path.basename(ckpt)}]")


In [ ]:
# ── Train ────────────────────────────────────────────────────────────────────
if initial_epoch >= EPOCHS:
    print(f"Already trained for {initial_epoch} epochs — nothing to do. Skip to next cell.")
else:
    callbacks = [
        DriveCheckpoint(SSL_DIR, SSL_HISTORY, history_acc, EPOCHS),
    ]
    ssl_model.fit(
        train_ds,
        validation_data=val_ds,
        steps_per_epoch=steps_per_epoch,
        validation_steps=val_steps,
        epochs=EPOCHS,
        initial_epoch=initial_epoch,
        callbacks=callbacks,
        verbose=2,
    )


In [ ]:
# ── Save final weights and summarise ─────────────────────────────────────────
ssl_model.save_weights(SSL_FINAL)
print(f"Saved final SSL encoder to {SSL_FINAL}")
print(f"  size: {os.path.getsize(SSL_FINAL) / 1e6:.2f} MB")

# Final history summary
with open(SSL_HISTORY) as f:
    h = json.load(f)
print(f"\nFinal SSL stats over {len(h['loss'])} epochs:")
print(f"  train loss: first={h['loss'][0]:.6f}  last={h['loss'][-1]:.6f}  best={min(h['loss']):.6f}")
print(f"  val   loss: first={h['val_loss'][0]:.6f}  last={h['val_loss'][-1]:.6f}  best={min(h['val_loss']):.6f}")


In [ ]:
# ── Sanity: reconstruction quality on a held-out batch ──────────────────────
val_batch = next(iter(val_ds))
inputs, target = val_batch
recon = ssl_model.predict(inputs, verbose=0)["pose_recon"]
mse  = float(tf.reduce_mean(tf.square(recon - target)).numpy())
mse0 = float(tf.reduce_mean(tf.square(inputs["pose_masked"][..., :3] - target)).numpy())
print(f"Reconstruction MSE on a val batch: {mse:.6f}")
print(f"  (compare to MSE between masked-input and target: {mse0:.6f})")
print(f"  improvement ratio: {mse0 / max(mse, 1e-9):.2f}x")
print()
print("Done. Move on to v5_supervised_train.ipynb")
